# Session 02 — Prepare data and explain group summaries

Revision ID: S02-r1

**Question:** How can an incident export become an analysis table without hiding quality problems?

## Learning outcomes

1. Select records and derive a variable while preserving raw data.
2. Justify separate treatments of duplicates, missing values, and invalid values.
3. Report group summaries with record and observed-value counts.
4. Communicate units, missingness, and limits of a descriptive comparison.

## Setup and workflow
Open this notebook in Jupyter or upload it in Colab. Run from the top. The same
CSV as Session 1 is embedded; no network or data upload is required. You may
instead upload the downloadable CSV and load it by its filename. Only pandas
and NumPy are needed; no random operations occur. Allow 20 minutes for the
worked demonstrations and 30 for exercises. Replace runnable placeholders and
write explanations; a successful Run All alone is not completion.

Keep raw data unchanged. Every correction below is a documented policy for this
synthetic example, not a universal recipe for all datasets.

In [1]:
import pandas as pd
import numpy as np
from io import StringIO
print("pandas", pd.__version__, "NumPy", np.__version__)

pandas 3.0.6 NumPy 2.5.3


## Provenance and dictionary

Synthetic course dataset, offered under CC0 1.0. It contains no real observations and is not a random sample or randomized deployment. Blanks are unknown logged durations of closed incidents, not zero and not open incidents.

# Data dictionary

Intended observational unit: a fictional closed software incident.
File: `software_incidents.csv`, UTF-8, comma-separated, header present.
Blank fields mean unrecorded information. Valid values describe the intended
schema; the raw export includes deliberate quality defects.

| Variable | Meaning | Statistical type | Units / valid values |
|---|---|---|---|
| `incident_id` | Incident identifier | Nominal identifier | `INC-001` through `INC-024`; unique after deduplication |
| `service` | Service affected | Nominal categorical | `api`, `web`, `worker`; may be blank |
| `severity` | Ordered operational severity | Ordinal categorical | `low` < `medium` < `high`; distances are not numeric |
| `resolution_hours` | Elapsed opening-to-closure time | Numerical, conceptually continuous | Hours, nonnegative; blank = unknown |
| `deploy_version` | Version at incident opening | Nominal categorical | `v1`, `v2`; not randomized treatment |
| `customer_impact` | Recorded customer impact | Binary categorical | `yes`, `no` |

Rounding duration to whole hours does not make it a count.
Derived in Session 2: `resolution_days = resolution_hours / 24`.
A display label `unknown` for missing service does not recover the true service.
In grouped summaries, `size` counts records; `count` counts nonmissing durations.


In [2]:
csv_text = 'incident_id,service,severity,resolution_hours,deploy_version,customer_impact\nINC-001,api,low,2,v1,no\nINC-002,web,medium,5,v1,yes\nINC-003,worker,high,8,v1,no\nINC-004,api,low,3,v1,yes\nINC-005,web,medium,,v1,no\nINC-006,worker,high,12,v1,yes\nINC-007,api,low,4,v1,no\nINC-008,web,medium,-2,v1,yes\nINC-009,worker,high,9,v1,no\nINC-010,api,low,2,v1,yes\nINC-011,,medium,8,v1,no\nINC-012,worker,high,15,v1,yes\nINC-013,api,low,3,v2,no\nINC-014,web,medium,6,v2,yes\nINC-015,worker,high,10,v2,no\nINC-016,api,low,4,v2,yes\nINC-017,web,medium,,v2,no\nINC-018,worker,high,18,v2,yes\nINC-019,api,low,5,v2,no\nINC-020,web,medium,10,v2,yes\nINC-021,worker,high,14,v2,no\nINC-022,api,low,6,v2,yes\nINC-023,web,medium,11,v2,no\nINC-024,worker,high,24,v2,yes\nINC-003,worker,high,8,v1,no\n'
raw = pd.read_csv(StringIO(csv_text))
raw_snapshot = raw.copy(deep=True)
raw.head()

,incident_id,service,severity,resolution_hours,deploy_version,customer_impact
0,INC-001,api,low,2.0,v1,no
1,INC-002,web,medium,5.0,v1,yes
2,INC-003,worker,high,8.0,v1,no
3,INC-004,api,low,3.0,v1,yes
4,INC-005,web,medium,NaN,v1,no


## Predict before cleaning
What would removing every row with any blank field do to a service comparison?
Write a prediction before running the checks. Missing, invalid, and duplicated
records are distinct issues and can require different decisions.

In [3]:
display(raw.isna().sum())
print("Exact repeated rows:", raw.duplicated().sum())
display(raw.loc[raw["resolution_hours"] < 0])

incident_id         0
service             1
severity            0
resolution_hours    2
deploy_version      0
customer_impact     0
dtype: int64

Exact repeated rows: 1


,incident_id,service,severity,resolution_hours,deploy_version,customer_impact
7,INC-008,web,medium,-2.0,v1,yes


**Interpret:** Which check reports unknown values, and which finds a present but impossible value? An exact duplicate can overweight an incident; a missing service does not make its duration unusable.

## Worked example: selection and units
This separate toy export has an exact repeat, an unknown value, and a logging
error. Predict the effect of each correction before running it. `.loc` selects
rows by a condition; division by 24 acts on every element without a Python loop.

In [4]:
toy = pd.DataFrame({"id": ["A", "B", "C", "A"], "hours": [24, np.nan, -4, 24]})
display(toy.loc[toy["hours"] > 0])
display(toy.assign(days=toy["hours"] / 24))

,id,hours
0,A,24.0
3,A,24.0


,id,hours,days
0,A,24.0,1.000000
1,B,NaN,NaN
2,C,-4.0,-0.166667
3,A,24.0,1.000000


24 hours is one day; a negative value remains invalid after conversion. Selection is not the same as correcting the source. **Predict:** How many distinct records and usable durations will remain?

In [5]:
toy_clean = toy.drop_duplicates().copy()
toy_clean.loc[toy_clean["hours"] < 0, "hours"] = np.nan
print("Records:", len(toy_clean))
print("Observed durations:", toy_clean["hours"].count())
print("Mean observed hours:", toy_clean["hours"].mean())

Records: 3
Observed durations: 1
Mean observed hours: 24.0


Three units remain, but only one duration supports the mean of 24 hours. That mean does not establish the average for the unknown durations. **Explain:** Why is replacing the two unknown times with zero a substantive assumption?

## Worked pipeline on the shared export
Policy: remove only exact export duplicates after inspection. An ID collision
with different content would require investigation. Convert negative durations
to missing, retaining the incident. Label missing service as `unknown` for
grouping; do not invent a true service or a numerical duration.

In [6]:
display(raw.loc[raw.duplicated(keep=False)])
clean = raw.drop_duplicates().copy()
assert clean["incident_id"].is_unique, "Investigate conflicting records with the same ID"
invalid = clean["resolution_hours"] < 0
clean.loc[invalid, "resolution_hours"] = np.nan
clean["service"] = clean["service"].fillna("unknown")

,incident_id,service,severity,resolution_hours,deploy_version,customer_impact
2,INC-003,worker,high,8.0,v1,no
24,INC-003,worker,high,8.0,v1,no


In [7]:
clean = clean.assign(resolution_days=clean["resolution_hours"] / 24)
display(clean.sort_values("resolution_hours").head())
print("Raw rows:", len(raw), "Clean records:", len(clean))
print("Invalid durations changed to missing:", int(invalid.sum()))

,incident_id,service,severity,resolution_hours,deploy_version,customer_impact,resolution_days
0,INC-001,api,low,2.0,v1,no,0.083333
9,INC-010,api,low,2.0,v1,yes,0.083333
3,INC-004,api,low,3.0,v1,yes,0.125000
12,INC-013,api,low,3.0,v2,no,0.125000
15,INC-016,api,low,4.0,v2,yes,0.166667


Raw rows: 25 Clean records: 24
Invalid durations changed to missing: 1


**Interpret:** The row-count change documents removal, while invalid-to-missing conversion changes a value without removing its incident. The `unknown` label records uncertainty; it does not resolve it.

In [8]:
service_summary = clean.groupby("service", dropna=False).agg(
    records=("incident_id", "size"),
    observed=("resolution_hours", "count"),
    mean_hours=("resolution_hours", "mean"),
)
service_summary["missing"] = service_summary["records"] - service_summary["observed"]
service_summary.round(2)

,records,observed,mean_hours,missing
service,,,,
api,8,8,3.62,0
unknown,1,1,8.00,0
web,7,4,8.00,3
worker,8,8,13.75,0


`size` counts records; `count` excludes missing values; `mean` uses available durations. For web, four of seven incidents contribute to the mean of 8 hours. Compare this denominator with other groups before interpreting differences. **Explain:** What could change if the missing times were systematically longer?

In [9]:
assert raw.equals(raw_snapshot), "The raw export must remain unchanged"
assert len(clean) == 24
assert clean["resolution_hours"].count() == 21
print("Raw data preserved; 24 distinct incidents, 21 observed valid durations.")

Raw data preserved; 24 distinct incidents, 21 observed valid durations.


## Optional extension: a different exclusion policy
`complete = clean.dropna(subset=["resolution_hours"])` creates a separate
duration-only subset. It is not needed for `mean`, which skips missing values.
Do not use the subset size as the total incident count. Compare what information
is lost before choosing an analysis-specific exclusion policy.

## S02-E1

Select the worker service into worker_only without changing clean. Add resolution_days from hours, then sort by resolution_days. Explain the units and why missing time remains missing.

In [10]:
worker_only = None
# TODO: use .loc, .copy(), .assign(), and .sort_values().

**Your response:**

_Write your explanation here._

## S02-E2

Create a short cleaning log: raw row count, exact repeats removed, remaining records, invalid durations changed to missing, and final available durations. Explain why the numerical duration column was not filled with zeros.

In [11]:
cleaning_log = None
# TODO: assemble counts from raw, invalid, and clean.

**Your response:**

_Write your explanation here._

## S02-E3

Group clean by deploy_version and report records, observed durations, and mean_hours. Interpret the difference using units and denominators. Give two reasons the result cannot establish a causal version effect.

In [12]:
version_summary = None
# TODO: adapt the service_summary example to deploy_version.

**Your response:**

_Write your explanation here._

## Save and communicate
Submit your completed E1–E3 notebook and assessment Q1–Q4. Restart and run all
cells before submission. A cleaning log explains which observations contribute
to each result. A group mean describes available values, not a causal effect.

## Readings
- [Learning Statistics with Python, Chapter 7](https://ethanweed.github.io/pythonbook/03.03-pragmatic_matters.html)
- [pandas groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)
- [pandas drop_duplicates](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html)
- [pandas fillna](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html)